# BumbaLM — ablação de negativos no Colab (A100 80 GB, LoRA)

Treina e avalia as três variantes do TCC (**nenhum**, **difíceis**, **aleatórios**) e o modelo base, com os dados preparados no node2.

**Antes da primeira sessão** (uma vez só, veja `colab/README.md`):
1. No node2: `bash node2/exportar_para_colab.sh`. Ele gera `processed_tjrs100k.tar.gz`, `pip_freeze_node2.txt`, `modelo_revisao_node2.txt` e `git_node2.txt`.
2. Suba esses arquivos para o Drive em `MyDrive/BumbaLM_colab/dados/`.
3. *Ambiente de execução → Alterar o tipo de ambiente de execução → **A100 GPU***, com RAM alta ligada.

**Em toda sessão** (inclusive depois de uma queda): rode as células 1 a 4 e depois a célula da etapa em que você está.
O treino retoma sozinho do último checkpoint no Drive, e as variantes concluídas são puladas.
Deixe a aba aberta: o Colab Pro desconecta abas ociosas e corta sessões de ~24 h.

## 1. GPU e Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 2. Código

Clona a branch do GitHub. Se o repositório for privado, crie o segredo `GITHUB_TOKEN` (ícone da chave, à esquerda) com um *token* de leitura.
Sem GitHub, suba um `codigo.zip` do repositório para `MyDrive/BumbaLM_colab/` e ponha `USAR_ZIP = True`.

As mudanças da pasta `colab/` precisam estar no commit dessa branch.

In [ ]:
import os, subprocess, zipfile
REPO_GITHUB = "alekxDias/BumbaLM-Embedding-4B-Train"
BRANCH = "tcc/ablacao-negativos-tjrs-h200"
USAR_ZIP = False
DESTINO = "/content/BumbaLM"
DRIVE = "/content/drive/MyDrive/BumbaLM_colab"

if USAR_ZIP:
    if not os.path.isdir(DESTINO):
        with zipfile.ZipFile(f"{DRIVE}/codigo.zip") as z:
            z.extractall("/content/BumbaLM_zip")
        raiz = next(r for r, d, f in os.walk("/content/BumbaLM_zip") if "colab" in d and "src" in d)
        os.rename(raiz, DESTINO)
elif os.path.isdir(f"{DESTINO}/.git"):
    subprocess.run(["git", "-C", DESTINO, "pull", "--ff-only"], check=True)
else:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    url = f"https://{token}@github.com/{REPO_GITHUB}.git" if token else f"https://github.com/{REPO_GITHUB}.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, DESTINO], check=True)

os.chdir(DESTINO)
os.environ["BUMBALM_DRIVE"] = DRIVE
!git log --oneline -1 2>/dev/null || echo "(codigo do zip)"

## 3. Dependências

Fixa os pacotes nas versões do node2 (`pip_freeze_node2.txt`), quando o arquivo existe. O `torch` do Colab é mantido.

In [ ]:
import os, subprocess, sys
PACOTES = ["sentence-transformers", "transformers", "peft", "accelerate", "datasets", "huggingface-hub"]
congelado = f"{DRIVE}/dados/pip_freeze_node2.txt"
versoes = {}
if os.path.isfile(congelado):
    for linha in open(congelado, encoding="utf-8", errors="replace"):
        if "==" in linha:
            nome, ver = linha.strip().split("==", 1)
            versoes[nome.lower().replace("_", "-")] = ver
pedidos = [f"{p}=={versoes[p]}" if p in versoes else p for p in PACOTES]
if "sentence-transformers" not in versoes:
    pedidos = ["sentence-transformers>=5.0,<6", "transformers>=4.51", "peft>=0.15", "accelerate>=1.0", "datasets>=3.0", "huggingface-hub"]
print("Instalando:", " ".join(pedidos))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pedidos, "pyyaml", "psutil", "pyarrow"], check=True)

## 4. Preparar a sessão

Faz três coisas e pula o que já estiver pronto:
- cria `configs/` no Drive (na primeira vez);
- descompacta os dados no disco local e confere o SHA-256 com o node2;
- baixa o modelo na mesma revisão do node2.

In [ ]:
!python colab/rodar.py preparar
!python colab/rodar.py checar

## 5. Calibração (uma vez, ~1–2 h)

- **`testar_modelo`** (opcional, ~5 min): confere que o modelo do treino é igual ao do Hub.
- **`calibrar`**: roda 10 passos de *difíceis* em 3 configurações de memória e 10 passos de *nenhum* com a validação do passo 0. Com isso:
  - confere o MRR@10 do modelo base, que tem que dar ~0,5;
  - monta a tabela **N tuplas × horas × unidades**.

Confira a taxa da A100 no painel *Recursos* e ajuste `--unidades_por_hora`.

In [ ]:
!python colab/rodar.py testar_modelo

In [ ]:
!python colab/rodar.py calibrar --unidades_por_hora 8.88

## 6. Fixar N e as frequências (uma vez, depois da calibração)

Preencha com a linha escolhida da tabela. A célula grava na cópia do Drive (`configs/config_train.yaml`) e mantém os comentários.

Depois que a primeira variante começar, **não mude mais**: o `rodar.py treinar` recusa hiperparâmetros diferentes entre as variantes.

In [ ]:
import re
N_TUPLAS = None        # ex.: 50000  (None = nao mexe)
SAVE_STEPS = None      # coluna 'save' da tabela
EVAL_STEPS = None      # coluna 'eval' da tabela (multiplo de SAVE_STEPS)
MINI_BATCH = None      # da calibracao (so memoria)
GRAD_CHECKPOINTING = None  # True/False, da calibracao (so memoria)

caminho = f"{DRIVE}/configs/config_train.yaml"
texto = open(caminho, encoding="utf-8").read()
def trocar(chave, valor):
    global texto
    if valor is None:
        return
    valor_yaml = str(valor).lower() if isinstance(valor, bool) else str(valor)
    texto, n = re.subn(rf"(?m)^(\s+{chave}:)\s*[^#\n]*", rf"\g<1> {valor_yaml}", texto, count=1)
    assert n == 1, f"chave {chave} nao encontrada"
    print(f"{chave}: {valor_yaml}")
trocar("limite_tuplas", N_TUPLAS)
trocar("save_steps", SAVE_STEPS)
trocar("eval_steps", EVAL_STEPS)
trocar("mini_batch_size", MINI_BATCH)
trocar("gradient_checkpointing", GRAD_CHECKPOINTING)
open(caminho, "w", encoding="utf-8").write(texto)
!python colab/rodar.py checar 2>&1 | sed -n '/Configuracao do treino/,/Disco local/p'

## 7. Treino: uma variante por vez

Comece pela **nenhum**: é a mais barata e valida o fluxo inteiro.
Na primeira vez, interrompa depois do primeiro checkpoint (*Ambiente → Interromper*) e rode a célula de novo. Ela tem que dizer "Retomando 'nenhum' de checkpoint-…" e continuar do mesmo passo.

Se a sessão cair: rode as células 1 a 4 e depois a mesma célula do treino.

In [ ]:
!python colab/rodar.py treinar nenhum

In [ ]:
!python colab/rodar.py treinar dificeis

In [ ]:
!python colab/rodar.py treinar aleatorios

## 8. Avaliação no teste completo

Avalia o modelo base e as variantes concluídas: 9.999 consultas e 92.101 trechos limpos. Grava os arquivos por consulta para o teste pareado.
Dá para rodar depois de cada variante: os modelos já avaliados são reaproveitados.

In [ ]:
!python colab/rodar.py avaliar

## 9. Andamento e resultados

In [ ]:
!python colab/rodar.py status

## 10. BM25 e teste de significância

O BM25 roda em CPU no notebook `notebooks/bm25_baseline_teste_colab.ipynb`. A seção 9 dele junta os arquivos por consulta (`por_consulta_test_*.jsonl`) para o bootstrap pareado.
Os arquivos dos modelos neurais ficam em:

`MyDrive/BumbaLM_colab/experiments/tcc_tjrs100k_ablacao_negativos_colab_a100/avaliacoes/test/<modelo>/`